# MNIST

In [ ]:
import idx2numpy
import numpy as np

## act.py

In [ ]:
class act:

    def funcio_act(y):
        return np.maximum(0, y)

    def dFuncio_act(x):
        x[x<=0] = 0
        x[x>0] = 1
        return x

## hiddenlayer.py

In [ ]:
class hiddenlayer:
    x: np.ndarray
    w: np.ndarray
    b: np.ndarray
    dw: np.ndarray
    db: np.ndarray
    z: np.ndarray
    N: np.ndarray
    alpha: float

    # Pido la tupla de w para crear una matriz de pesos iniciales de la medida que me diga la tupla
    def __init__(self, w_tuple: list, b: int):
        self.w = np.random.rand(*w_tuple)
        self.b = np.random.rand(b)
        self.alpha = 0.01

    # Esta funcion sirve para calcular z / y a partir de xw + B o Nw + B donde x y N son un vector, w es una matriz y B es un vector
    def forward(self, x: np.ndarray):
        self.x = x
        self.z = x@self.w + self.b
        self.N = act.funcio_act(self.z)
        return self.N
    
    # dy es el coste que se quiere propagar a esta capa y devolvemos
    def backward(self, err):
        err = err*act.dFuncio_act(self.z)
        self.dw = self.x.T@err
        self.db = np.sum(err, axis=0)
        self.update()
        return err @ self.w.T

    # Esta funcion sirve para actualizar las w o B de la capa (layer) e ir reduciendo el coste o loss del modelo
    def update(self):
        self.w = self.w - self.alpha*self.dw
        self.b = self.b - self.alpha*self.db
        return

## model.py

In [ ]:
class model:
    w_shape: list
    b_shape: list
    list_hidden_layers: list[hiddenlayer]

    # Le paso una lista de tuplas para inicializar w y una lista de ints para b en cada capa oculta que se vaya a crear
    def __init__(self, w_shape: list, b_shape: list):
        self.w_shape = w_shape
        self.b_shape = b_shape
        self.list_hidden_layers = []

        # Inicializo las capas ocultas a partir de un bucle y las guardo en la lista
        for i in range(len(w_shape)):
            self.list_hidden_layers.append(hiddenlayer(w_shape[i], b_shape[i]))
        pass

    # En esta función obtengo la y predicha haciendo un forward por cada layer que tengo
    # Paso por la hidden_layer (esto ya es una y, pero no la final o predicha que busco)
    # Necesito pasar la cantidad de capas para que se repita, tambien los datos de entreno
    # Luego aplico la funcion de activacion para obtener la siguiente N en caso de que hayan mas capas o la y predicha si ya no hay mas capas
    def forward(self, x: np.ndarray):
        # Recorro la lista de capas ocultas para hacer el calculo en cada capa oculta
        for hidden_layer in self.list_hidden_layers:
            N = hidden_layer.forward(x)
            x = N
        return N

    #En esta funcion obtengo los dw, dB y dN para hacer update de w y b a las hidden_layer 
    def backward(self, loss_gradient):
        # Recorro la lista de capas ocultas desde el final
        for hidden_layer in reversed(self.list_hidden_layers):
            loss_gradient = hidden_layer.backward(loss_gradient)
        pass

    # En esta funcion obtengo el coste o loss del modelo, el cual debo propagar a las capas anteriores para ir calculando las dw, dB y dN / y (esta y se sigue propagando atrás)
    def loss(self, y: np.ndarray, y_train: np.ndarray):
        return (y - y_train)

## data.py

In [ ]:
class data:

    # Estos atributos debo inicializarlos en el main pasandoles este mismo valor (debo crear un constructor)
    x_train: np.ndarray
    y_train: np.ndarray
    x_test: np.ndarray
    y_test: np.ndarray

    def __init__(self, x_train, x_test, y_train, y_test):
        self.x_train = x_train
        self.x_test = x_test
        self.y_train = np.eye(10)[y_train]
        self.y_test = np.eye(10)[y_test]
        pass

    # En esta funcion debo dividir con reshape las x entrenadas en n cantidad para no entrenar todas directamentes (ineficaz)
    # Por lo tanto tendré diferentes etapas y cada etapa será un entrenamiento para el modelo, haciendo que en cada etapa actualice las w y B de cada capa
    # Entonces aqui creo un modelo y llamo a la funcion forward del modelo por cada etapa hecha, calculo el coste y luego llamo al backward
    def trainloop(self, m: model):
        # Hago un reshape de los datos de entreno para tener los datos como un solo vector, haciendo que sean indexables (estos datos se siguen tratando de la misma manera)
        x_train_batched = self.x_train.reshape(60000, 784)
        # Hago un loop en el que en cada iteracion se hace un entreno con 500
        for i in range(0, len(x_train_batched), 500):
            x_batch = x_train_batched[i:i+500]
            y_batch = self.y_train[i:i+500]

            y = m.forward(x_batch)
            loss = m.loss(y, y_batch)
            m.backward(loss)
        return loss

## main.py (ejecución)

In [ ]:
w_shape = [(784, 32), (32, 10)]
b_shape = [32, 10]

m = model(w_shape, b_shape)

x_train = idx2numpy.convert_from_file('./train-images.idx3-ubyte')
y_train = idx2numpy.convert_from_file('./train-labels.idx1-ubyte')
x_test = idx2numpy.convert_from_file('./t10k-images.idx3-ubyte')
y_test = idx2numpy.convert_from_file('./t10k-labels.idx1-ubyte')

d = data(x_train, x_test, y_train, y_test)

print(d.trainloop(m))